<a href="https://colab.research.google.com" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background:linear-gradient(135deg,#1a237e,#1565c0);
     padding:36px 32px 28px; border-radius:10px; margin-bottom:4px;">
<h1 style="color:#fff;font-family:'Georgia',serif;font-size:2em;margin:0 0 6px 0;">
    Na Emission Spectrum</h1>
<h3 style="color:#90caf9;font-family:'Georgia',serif;font-weight:normal;
           font-size:1.15em;margin:0 0 14px 0;">
    Chemistry 76 &nbsp;·&nbsp; Peak Finding
</h3>
<div style="color:#bbdefb;font-size:0.9em;line-height:1.8;">
    <b style="color:#fff;">What this notebook does:</b>
    Loads your raw scan, identifies all emission peaks, separates Hg calibration
    lines from Na candidates, removes known lamp impurities, and delivers two
    CSV files: a Na peak list (raw observed wavelengths only) and the Hg
    calibration pairs. The wavelength corrections and wavenumber conversion
    are yours to do in your analysis.
</div></div>


## How to use this notebook

Run every cell **in order** using **Shift + Enter**.

At the end, two files are saved to your Google Drive and downloaded:
- `na_peaks.csv` — your Na emission peaks (λ_obs in Å, intensity in V)
- `hg_peaks.csv` — the Hg calibration pairs (λ_obs observed + λ_true literature)

> Do not change any constants or the data URL.


## Step 1 — Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import find_peaks
from scipy.stats import linregress
from scipy.optimize import brentq
from IPython.display import display
import os

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelsize': 11, 'axes.titlesize': 12,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'font.family': 'serif',
})
print("✓  Libraries loaded.")


## Step 2 — Pre-configured constants

All values below are **locked**. Do not change them.

In [ ]:
PEAK_HEIGHT    = 0.20
MIN_SEPARATION = 2.0
HG_MATCH_TOL   = 5.0
BOGUS_TOL      = 4.0

R_NA   = 109734.698
A_CORR = 2.6343e-4
B_CORR = 0.07621

HG_TRUE = np.array([3125.67, 3131.55, 3650.15, 4046.56,
                     4358.33, 5460.74, 5769.60, 5790.66])

BOGUS = np.array([
    # Standard lamp impurities (lab manual):
    3342, 4159, 4164, 4182, 4191, 4199, 4201, 4252, 4260,
    4267, 4272, 4300, 4316, 4334, 4345, 4366, 4425, 4511,
    4522, 4554, 4596, 4629, 4703, 4935, 5163, 5188, 5222,
    5496, 5559, 5573, 5607, 5651,
    # Secondary Hg multiplet lines (not used for calibration):
    3024.0, 3027.0, 3135.0, 3655.0, 3663.0, 4078.0,
    # Confirmed Ne contamination:
    5078.5, 5852.5, 5944.8,
    # Bright lamp line adjacent to D-lines — not a ground-state Na transition:
    5899.2,
])

NEAR_HG = np.array([5758., 5760., 5762., 5764., 5766.,
                     5793., 5795., 5797., 5799., 5801.])

ZONE_LO = 4100.0;  ZONE_HI = 4640.0
WHITELIST_ZONE = np.array([
    4081.0, 4151.1, 4153.1, 4170.2, 4176.5, 4214.6, 4217.6,
    4241.6, 4243.6, 4277.1, 4322.2, 4327.5, 4349.9,
    4392.7, 4396.7, 4496.0, 4503.8,
])
I_UV  = 0.20;  I_MID = 0.35;  I_NIR = 0.25

print(f"✓  Constants: BOGUS = {len(BOGUS)} entries, R_Na = {R_NA} cm\u207b\u00b9")


## Step 3 — Load your raw data

In [ ]:
url     = "https://drive.google.com/file/d/1kyrYdyCD-8IXJdRQSJ_ZZJXyjc1erZBT/view?usp=drive_link"
file_id = url.split('/')[-2]
dl_url  = f"https://drive.google.com/uc?id={file_id}"

dataframe  = pd.read_csv(dl_url, names=["Wavelength (A)", "Intensity"], sep='\t')
data       = dataframe.to_numpy()
wavelength = data[:, 0]
intensity  = data[:, 1]

print(f"✓  Data loaded: {len(wavelength):,} points")
print(f"   Range : {wavelength[0]:.1f} – {wavelength[-1]:.1f} Å")
print(f"   Signal: {intensity.min():.4f} – {intensity.max():.4f} V")
print(f"   Step  : {np.mean(np.diff(wavelength)):.4f} Å/point")

## Step 4 — Raw spectrum overview

Check: flat baseline ~0.05–0.08 V, bright peaks saturating at 10 V.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 3.5),
                                gridspec_kw={'width_ratios': [3,1]})
ax1.plot(wavelength, intensity, color='#1e3a5f', lw=0.5, alpha=0.85, rasterized=True)
ax1.axhline(PEAK_HEIGHT, color='#e53935', lw=0.8, ls='--', label=f'Threshold ({PEAK_HEIGHT} V)')
ax1.set_xlabel('Observed Wavelength (Å)'); ax1.set_ylabel('Signal (V)')
ax1.set_title('Raw Spectrum'); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.2)
ax1.set_xlim(wavelength[0], wavelength[-1])
background = intensity[intensity < PEAK_HEIGHT]
ax2.hist(background, bins=50, color='#1565c0', alpha=0.7, edgecolor='white', lw=0.3)
ax2.axvline(PEAK_HEIGHT, color='#e53935', lw=1.5, ls='--')
ax2.set_xlabel('Intensity (V)'); ax2.set_ylabel('Count')
ax2.set_title(f'Background: mean = {background.mean():.4f} V')
plt.tight_layout(); plt.show()


## Step 5 — Automatic peak detection

Finds every local maximum ≥ 0.20 V with separation ≥ 2.0 Å.

In [ ]:
step         = np.mean(np.diff(wavelength))
min_dist_pts = int(MIN_SEPARATION / step)
peak_idx, _  = find_peaks(intensity, height=PEAK_HEIGHT, distance=min_dist_pts)
pk_wl        = wavelength[peak_idx]
pk_sig       = intensity[peak_idx]
print(f"Step size:   {step:.4f} Å/point")
print(f"Peaks found: {len(pk_wl)} above {PEAK_HEIGHT} V")


## Step 6 — Separate Hg and Na peaks

Any peak within 5 Å of a literature Hg wavelength is a calibration line. You will apply the calibration fit in your analysis.

In [ ]:
nearest_hg = np.array([np.min(np.abs(w - HG_TRUE)) for w in pk_wl])
is_hg      = nearest_hg < HG_MATCH_TOL
hg_obs_wl  = pk_wl[is_hg];   hg_obs_sig = pk_sig[is_hg]
na_wl_all  = pk_wl[~is_hg];  na_sig_all = pk_sig[~is_hg]

print(f"Hg calibration peaks : {is_hg.sum()}")
print(f"Na candidates (raw)  : {(~is_hg).sum()}")
print()
print(f"  {'Detected λ_obs (Å)':>20}  {'HG_TRUE (Å)':>12}  {'Δλ (Å)':>8}")
print("  " + "─"*44)
for w in sorted(hg_obs_wl):
    t = HG_TRUE[np.argmin(np.abs(w - HG_TRUE))]
    print(f"  {w:>20.3f}  {t:>12.2f}  {w-t:>+8.3f}")


## Step 7 — Remove impurities and apply zone filter

Impurities, secondary Hg lines, Ne contamination lines, and one unassignable lamp line at 5899.2 Å are removed. The dense 4100–4640 Å region is filtered to a whitelist of known Na assignments.

In [ ]:
is_bogus   = np.array([np.min(np.abs(w - BOGUS)) < BOGUS_TOL for w in na_wl_all])
is_near_hg = np.array([np.min(np.abs(w - NEAR_HG)) < 3.0      for w in na_wl_all])

# D-line protection: prevents BOGUS window around 5899.2 from removing D1 at 5895.96
is_dline_all = (na_wl_all > 5880) & (na_wl_all < 5910)
is_bogus[is_dline_all & (na_wl_all < 5898)] = False

in_zone  = (na_wl_all >= ZONE_LO) & (na_wl_all <= ZONE_HI)
is_dline = (na_wl_all > 5880) & (na_wl_all < 5910)
near_wl  = np.array([np.min(np.abs(w - WHITELIST_ZONE)) < 6.0 for w in na_wl_all])

uv_r  =  na_wl_all < 3500
mid_r = ((na_wl_all >= 3500) & (na_wl_all < 4100)) | \
        ((na_wl_all >= 4640) & (na_wl_all < 5600))
nir_r =  na_wl_all >= 5600

keep_s = np.zeros(len(na_wl_all), dtype=bool)
keep_s[is_dline]              = True
keep_s[~in_zone&~is_dline&uv_r]  = na_sig_all[~in_zone&~is_dline&uv_r]  >= I_UV
keep_s[~in_zone&~is_dline&mid_r] = na_sig_all[~in_zone&~is_dline&mid_r] >= I_MID
keep_s[~in_zone&~is_dline&nir_r] = na_sig_all[~in_zone&~is_dline&nir_r] >= I_NIR
keep_s[in_zone] = near_wl[in_zone]

is_clean     = ~is_bogus & ~is_near_hg & keep_s
na_wl_clean  = na_wl_all[is_clean]
na_sig_clean = na_sig_all[is_clean]

print(f"Na raw:           {len(na_wl_all)}")
print(f"  bogus/Ne:      -{is_bogus.sum()}")
print(f"  near-Hg:       -{(~is_bogus & is_near_hg).sum()}")
print(f"  zone filtered: -{(~is_bogus & ~is_near_hg & ~keep_s).sum()}")
print(f"  CLEAN:          {is_clean.sum()}  (expected: 54)")


## Step 8 — Annotated spectrum

- 🔴 **Red ▼** — Hg calibration lines
- 🟠 **Orange ▲** — your clean Na peaks
- ✗ **Grey** — excluded lines

The D-line zoom should show exactly **two** orange peaks near 5890–5896 Å.


In [ ]:
fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(2, 1, hspace=0.5, height_ratios=[3, 1.8])
ax1 = fig.add_subplot(gs[0]); ax2 = fig.add_subplot(gs[1])

ax1.plot(wavelength, intensity, color='#1e3a5f', lw=0.5, alpha=0.8, rasterized=True)
ax1.plot(hg_obs_wl, hg_obs_sig, 'v', color='#e53935', ms=9, zorder=5, label=f'Hg ({is_hg.sum()})')
ax1.plot(na_wl_clean, na_sig_clean, '^', color='#ff8f00', ms=7, zorder=4, label=f'Na ({is_clean.sum()})')
ax1.plot(na_wl_all[is_bogus | is_near_hg], na_sig_all[is_bogus | is_near_hg],
         'x', color='#9e9e9e', ms=5, zorder=3, label='Excluded')
ax1.axhline(PEAK_HEIGHT, color='gray', ls='--', lw=0.7)
for wl_h, sig_h in zip(hg_obs_wl, hg_obs_sig):
    t = HG_TRUE[np.argmin(np.abs(wl_h - HG_TRUE))]
    ax1.annotate(f'Hg\n{t:.0f}', xy=(wl_h, min(sig_h,10.5)), xytext=(0,10),
                 textcoords='offset points', fontsize=6.5, ha='center', color='#c62828',
                 arrowprops=dict(arrowstyle='-', color='#c62828', lw=0.5))
ax1.set_title(f'Full Spectrum — {is_hg.sum()} Hg lines, {is_clean.sum()} Na peaks', fontsize=12)
ax1.set_xlabel('λ_obs (Å)'); ax1.set_ylabel('Signal (V)')
ax1.legend(loc='upper left', fontsize=8); ax1.set_xlim(wavelength[0], wavelength[-1])
ax1.grid(True, alpha=0.15)

lo, hi = 5820, 5960
mask = (wavelength >= lo) & (wavelength <= hi)
ax2.plot(wavelength[mask], intensity[mask], color='#1e3a5f', lw=0.9)
na_z = (na_wl_clean >= lo) & (na_wl_clean <= hi)
ax2.plot(na_wl_clean[na_z], na_sig_clean[na_z], '^', color='#ff8f00', ms=10, zorder=4)
for w, s in zip(na_wl_clean[na_z], na_sig_clean[na_z]):
    ax2.annotate(f'{w:.2f}', xy=(w,s), xytext=(0,6), textcoords='offset points',
                 fontsize=9, ha='center', color='#37474f', fontweight='bold')
ax2.set_title('Na D-line region — should show exactly TWO peaks (~5893.68 and ~5895.96 Å)')
ax2.set_xlabel('λ_obs (Å)'); ax2.set_ylabel('Signal (V)')
ax2.set_xlim(lo, hi); ax2.grid(True, alpha=0.15)

plt.savefig('my_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: my_spectrum.png")


## Step 9 — Doublets and suspicious peaks

---

### Doublets

Every Na emission transition in this spectrum produces two closely spaced lines.
This happens because spin-orbit coupling splits each $nL$ level ($L > 0$) into
two sub-levels with total angular momentum $J = L \pm \tfrac{1}{2}$. Both the
upper and lower levels are split, so every transition produces a **doublet** —
two lines separated by typically 6–20 cm⁻¹.

**Identifying doublet pairs in your list** (do this after converting to wavenumbers):

Scan your sorted peak list for adjacent pairs satisfying all three criteria:
1. Separated by **8–25 cm⁻¹** in wavenumber
2. **Adjacent** in the sorted list — no unrelated peak between them
3. **Similar intensity** — both components are usually within a factor of ~2

**What to report for each doublet:**
$$\tilde{\nu}_\text{mean} = \frac{\tilde{\nu}_1 + \tilde{\nu}_2}{2} \qquad
\Delta\tilde{\nu}_\text{split} = |\tilde{\nu}_1 - \tilde{\nu}_2|$$

Use $\tilde{\nu}_\text{mean}$ for all Δ-table matching and line assignment.
Report $\Delta\tilde{\nu}_\text{split}$ as the spin-orbit energy splitting for
that orbital.

**Some peaks will have no resolvable partner** in this dataset. This can happen
because the partner component falls below the detection threshold, or because the
splitting is smaller than the instrument step size (~0.25 Å) and both components
appear as one broadened peak. When you cannot find a partner within 25 cm⁻¹,
use the single observed $\tilde{\nu}$ directly for assignment — do not spend
time looking for a partner that is not there.

---

### The Na D-lines — start here

The two brightest peaks in your list near 5890–5896 Å are the **sodium D-lines**:
the resolved doublet of the $3P \to 3S$ transition. Their splitting of \~6.6 cm⁻¹
is the spin-orbit splitting of the $3P$ level. These are your anchor — assign
them first, derive $\delta_{3S}$ from their mean wavenumber, and work outward
from there.

---

### Suspicious peaks — mark as unassigned

A small number of peaks in your list are real lamp emission features but will
not match any entry in the Δ table. When a peak has no plausible Δ-table match
after careful checking, mark it **unassigned** and move on — it does not affect
your determination of $R_\text{Na}$ or the ionization energy.


## Step 10 — Save and download

Files are saved to `My Drive / Chem76_Na_Lab /` and downloaded to your computer.
Run this cell last.


In [ ]:
sort_na = np.argsort(na_wl_clean)[::-1]

na_df = pd.DataFrame({
    'peak_num':     range(1, is_clean.sum() + 1),
    'lambda_obs_A': np.round(na_wl_clean[sort_na], 3),
    'intensity_V':  np.round(na_sig_clean[sort_na], 4),
})

hg_true_m = np.array([HG_TRUE[np.argmin(np.abs(w - HG_TRUE))] for w in hg_obs_wl])
seen = {}
for obs, true in zip(hg_obs_wl, hg_true_m):
    if true not in seen or abs(obs-true) < abs(seen[true][0]-true):
        seen[true] = (obs, true)
hg_pairs = sorted(seen.values(), key=lambda x: x[1])

hg_df = pd.DataFrame({
    'hg_line_num':   range(1, len(hg_pairs) + 1),
    'lambda_obs_A':  [round(o, 3) for o, t in hg_pairs],
    'lambda_true_A': [round(t, 2) for o, t in hg_pairs],
    'intensity_V':   [round(float(pk_sig[list(pk_wl).index(o)]), 4) for o, t in hg_pairs],
})

na_df.to_csv('na_peaks.csv', index=False)
hg_df.to_csv('hg_peaks.csv', index=False)

print(f"Na peaks : {len(na_df)}  (expected: 54)")
print(f"Hg lines : {len(hg_df)}  (expected: 7)")
print()
print(na_df.to_string(index=False))
print()
print(hg_df.to_string(index=False))
print()
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    import shutil
    save_dir = '/content/drive/MyDrive/Chem76_Na_Lab'
    os.makedirs(save_dir, exist_ok=True)
    shutil.copy('na_peaks.csv',    f'{save_dir}/na_peaks.csv')
    shutil.copy('hg_peaks.csv',    f'{save_dir}/hg_peaks.csv')
    shutil.copy('my_spectrum.png', f'{save_dir}/my_spectrum.png')
    print(f"✓  Saved to Google Drive: {save_dir}/")
    from google.colab import files
    files.download('na_peaks.csv')
    files.download('hg_peaks.csv')
    files.download('my_spectrum.png')
except ImportError:
    print("(Running locally — files saved in current directory)")


---

## Reference — Theory and Calculations

Everything you need to carry out the lab analysis, in one place.

---

### 1. Hg Wavelength Calibration

The monochromator has a small mechanical offset. Correct for it with a linear fit
using the Hg calibration pairs from `hg_peaks.csv`:

$$\lambda_\text{true} = m \cdot \lambda_\text{obs} + b$$

Fit $\lambda_\text{true}$ vs $\lambda_\text{obs}$ using linear regression.  
Check $R^2 > 0.9999$ and plot the residuals — all should be $< 0.5$ Å.

---

### 2. Air → Vacuum Wavelength Correction

Wavelengths from the monochromator are measured in air. Convert to vacuum:

$$\lambda_\text{vac} = \lambda_\text{true} + \left(2.6343\times10^{-4}\,\lambda_\text{true} + 0.07621\right) \quad [\text{Å}]$$

---

### 3. Wavelength → Wavenumber

$$\tilde{\nu}\ (\text{cm}^{-1}) = \frac{10^8}{\lambda_\text{vac}\ (\text{Å})}$$

Apply this to every Na peak. The column $\tilde{\nu}/R_\text{Na}$ (with
$R_\text{Na} = 109734.698$ cm$^{-1}$) is used for Δ-table matching.

---

### 4. The Rydberg Formula (Eq. 5)

Energy levels of the Na valence electron are hydrogen-like, corrected by a
quantum defect $\delta_L$ that accounts for core penetration:

$$W_n = -\frac{R_\text{Na}}{(n - \delta_L)^2}$$

The wavenumber of an emission transition from upper level $n_2$ to lower level $n_1$ is:

$$\tilde{\nu} = R_\text{Na}\!\left[\frac{1}{(n_1 - \delta_{L_1})^2} - \frac{1}{(n_2 - \delta_{L_2})^2}\right] \tag{5}$$

**Quantum defects for Na** (from lab handout Table I):

| Series | Transition | $\delta_L$ |
|---|---|---|
| P | $nP \to 3S$ | $\delta_P \approx 0.883$ |
| S | $nS \to 3P$ | $\delta_S \approx 0.063$ |
| D | $nD \to 3P$ | $\delta_D \approx 0.012$ |

The **3S ground state** has a large quantum defect $\delta_{3S} \approx 1.373$
because the 3s electron penetrates the core strongly. Derive it from the
D-line mean wavenumber using Eq. (5) inverted:

$$\delta_{3S} = 3 - \frac{1}{\sqrt{\dfrac{1}{(3-\delta_P)^2} + \dfrac{\tilde{\nu}_{D\text{-line}}}{R_\text{Na}}}}$$

---

### 5. The Δ-Table Method (Line Assignment)

For two consecutive members of the same series ($n$ and $n+1$, same lower level),
their wavenumber difference divided by $R_\text{Na}$ equals:

$$\frac{\tilde{\nu}_{n+1} - \tilde{\nu}_n}{R_\text{Na}} = \Delta_{n+1,n}(\delta_L)
= \frac{1}{(n-\delta_L)^2} - \frac{1}{(n+1-\delta_L)^2} \tag{8}$$

**Procedure:**

1. For every pair of peaks $(i,j)$ in your list, compute
   $(\tilde{\nu}_i - \tilde{\nu}_j)/R_\text{Na}$
2. Look up this value in the Δ table (lab handout)
3. The **row** gives $n$; the **column** gives $\delta_L$
4. The magnitude of $\delta_L$ identifies the series:
   - $\delta \approx 0.88$ → **P series** ($nP \to 3S$)
   - $\delta \approx 0.06$ → **S series** ($nS \to 3P$)
   - $\delta \approx 0.01$ → **D series** ($nD \to 3P$)
5. **Anchor first:** the D-line doublet (peaks 4 & 5,
   $\tilde{\nu} \approx 16958$–$16964$ cm$^{-1}$) is the $3P \to 3S$ transition.
   Use this to fix $\delta_{3S}$, then predict the rest of the P series.

---

### 6. Doublets — Mean Wavenumber and Spin-Orbit Splitting

Each line is a doublet ($J = L \pm \tfrac{1}{2}$). Use the **mean wavenumber**
for assignment and report the **splitting** as the spin-orbit energy:

$$\tilde{\nu}_\text{mean} = \frac{\tilde{\nu}_1 + \tilde{\nu}_2}{2}
\qquad
\Delta\tilde{\nu}_\text{SO} = |\tilde{\nu}_1 - \tilde{\nu}_2|$$

---

### 7. Experimental $R_\text{Na}$ from the D-Series

For the D series ($nD \to 3P$), Eq. (5) rearranges to:

$$\tilde{\nu} = T - \frac{R_\text{Na}}{(n-\delta_D)^2}$$

where $T$ is the ionization energy of the $3P$ level. Plot $\tilde{\nu}$ vs
$x = 1/(n-\delta_D)^2$ for $n = 4,5,6,\ldots$ — the slope is $-R_\text{Na}$
and the intercept is $T$.

$$\text{IP}(3S) = T + \tilde{\nu}_{3P\to 3S}^\text{mean}$$

---

### Key Constants

| Quantity | Value |
|---|---|
| $R_\text{Na}$ | 109734.698 cm$^{-1}$ |
| $\delta_P$ | 0.883 (n=3), 0.867 (n=4), 0.862 (n=5) |
| $\delta_S$ | 0.063 |
| $\delta_D$ | 0.012 |
| IP(Na) literature | 41449.65 cm$^{-1}$ = 5.1391 eV |
| Na D-line (lit.) | 16964.8 / 16958.2 cm$^{-1}$ |
